In [0]:
dbutils.widgets.text("catalog","ai")
dbutils.widgets.text("controlTableSchema","00_etlcontrol")
dbutils.widgets.text("env","dev")
dbutils.widgets.text("source","tutorial")


env = dbutils.widgets.get("env")
catalog = dbutils.widgets.get("catalog")
controlTableSchema = dbutils.widgets.get("controlTableSchema")
source = dbutils.widgets.get("source")


In [0]:
from pyspark.sql.functions import pandas_udf, col,expr, sha2, current_user, current_timestamp, lit, date_format, when, to_json, struct
from typing import Iterator
import pandas as pd
from pyspark.sql.types import ArrayType, StringType, BinaryType
import io, os, sys
from pypdf import PdfReader
from pyspark.sql import SparkSession

project_root = os.path.abspath("../../") 
if project_root not in sys.path:
    sys.path.append(project_root)

# Reload module to pick up any changes during development
import importlib
import lib.common.sparkConfig
importlib.reload(lib.common.sparkConfig)

import lib.bronze.autoloaderconfig
importlib.reload(lib.bronze.autoloaderconfig)

from lib.common.sparkConfig import SourceInitialize
from lib.bronze.autoloaderconfig import AutoLoaderInitialize



In [0]:
# from pyspark.sql.types import StructType, StructField, LongType, StringType

def getNotebookName():
        current_notebook = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        return current_notebook.split("/")[-1] if current_notebook.split("/") else current_notebook

spark = SparkSession.builder.appName(getNotebookName()).getOrCreate()
# spark.conf.set("spark.sql.streaming.minBatchesToRetain", "100")

eod_stuct = StructType([
        StructField("isvalid",BooleanType()),
        StructField("error", StringType())
])

class BronzeETL:
        
        def __init__(self, spark, src: SourceInitialize, ldr: AutoLoaderInitialize):
                self.spark = spark
                self.src = src
                self.autoLoaderConfigProperties = ldr.getAutoloaderConfigProperties()
                self.autoLoaderFileMetaProperties = ldr.getAutoloaderFileMetaProperties()
                self.autoloaderTagetTableProperties = ldr.getAutoloaderTagetTableProperties()
                self.autoloaderTagetPartitionColumn = ldr.getAutoloaderTagetPartitionColumn()
                self.sourceProperties = src.getSourceProperties()
                self.metaColumns = self.getFileMetaData()
                self.sourceName = src.getSourceName()
                self.loadid = src.getLoadId()
                self.rejectedRecordsTable = src.getRejectedRecordsTable()
                self.inline_threshold_bytes = self.sourceProperties.get("INLINE_THRESHOLD_BYTES")

        def getFileMetaData(self):

                return [f'{val} as {name}' for name,val in self.autoLoaderFileMetaProperties.items()]

        
        def applyFileRouting(self, batch_df):

                return (batch_df.select(
                        "*",
                        (when(col("length") <= self.inline_threshold_bytes, lit("INLINE")).otherwise(lit("EXTERNAL_POINTER"))).alias("externalpointer"),
                        (when(col("length") <= self.inline_threshold_bytes, col("content")).otherwise(lit(None).cast(BinaryType()))).alias("file_content")
                )
                .drop("content")
                .withColumnRenamed("file_content","content")
                .select("*",sha2(col("content"), 256).alias("documentKey"))
                )
                
        def file_size_validation(self, batch_df, batch_id):
                
                batch_df = (batch_df
                                .select("*", 
                                (when(col('length') == 0, "ZERO SIZE FILE")
                                .when(col('length').isNull(), "NULL FILE")
                                .otherwise("VALID FILE")).alias("isvalid")
                                ))
                
                batch_df = batch_df.filter(col("isvalid") == "VALID FILE")

                (batch_df.drop("content").select(lit("batch_id").alias("batch_id"),
                                to_json(struct("*")).alias("rejected_payload"),
                                lit(self.sourceName).alias("source_system"),
                                lit("ZERO SIZE FILE").alias("rejection_reason"),
                                current_timestamp().alias("rejected_at"),
                                current_user().alias("createdby"),
                                )
                        .write
                        .format("delta")
                        .mode("append")
                        .option("mergeSchema", "true")
                        .saveAsTable(f'{self.autoloaderTagetTableProperties.get("targetCatelog")}.{self.autoloaderTagetTableProperties.get("targetSchema")}.{self.autoloaderTagetTableProperties.get("layer")}_{self.rejectedRecordsTable}')
                        )
                
                return batch_df.filter(col("isvalid") == "VALID FILE")
                
                
        def end_of_file_validation(self, contact) -> dict:

                

                
                  



        def processBatch(self,batch_df, batch_id):

                # batch_df.persist()

                if batch_df.isEmpty():
                        return
                
                cnt = batch_df.select("path").count()                
                batch_df = batch_df.select("*", lit(batch_id).alias("batchid"), lit(self.loadid).alias("loadid"))
                batch_df = self.applyFileRouting(batch_df)
                batch_df = self.file_size_validation(batch_df, batch_id)
                (batch_df
                        .write
                        .format(f'{self.autoloaderTagetTableProperties.get("targetType")}')
                        .mode(f'{self.autoloaderTagetTableProperties.get("mode")}')
                        .option("mergeSchema", "true")
                        .saveAsTable(f'{self.autoloaderTagetTableProperties.get("targetCatelog")}.{self.autoloaderTagetTableProperties.get("targetSchema")}.{self.autoloaderTagetTableProperties.get("layer")}_{ self.sourceName}_document')
                                
                        )
                
                self.src.auditInsert(self.loadid, batch_id, cnt,self.autoloaderTagetTableProperties.get("targetCatelog"), self.autoloaderTagetTableProperties.get("targetSchema"), self.autoloaderTagetTableProperties.get("layer"))
                
                # batch_df.unpersist()    

        def bronzeLoad(self):

                
                query = (self.spark.readStream.format("cloudFiles") 
                 .options(**self.autoLoaderConfigProperties)
                 .load(self.sourceProperties.get("file_path"))
                 .selectExpr("*", *self.metaColumns)
                 .writeStream
                 .foreachBatch(self.processBatch)
                 .option("checkpointLocation", f"{self.sourceProperties.get('checkpint')}/{self.sourceName}")
                 .trigger(availableNow=True)
                 .start()
                )

                query.awaitTermination()
                

                
                
if __name__ == "__main__":



        print(source)
        src = SourceInitialize(spark, getNotebookName(), env, catalog, controlTableSchema, source)
        src.checkSetup()

        ldr = AutoLoaderInitialize(spark, catalog, controlTableSchema,src.getAutoloaderPropertiesTable(), src.getSourceId())
        ldr.checkSetup()

        bz_etl = BronzeETL(spark, src, ldr)        
        bz_etl.bronzeLoad()

In [0]:
df = (spark.read.table("ai.`02_bronze`.bronze_tutorial_document"))

display(df.drop("content"))

df = spark.read.table("ai.`02_bronze`.bronze_auditlog")
display(df)

